# How the post office sorts 20 million letters a night

Radix sort: no comparisons, one digit at a time, and every pass must be stable.

In [1]:
def radix_sort(codes, digits=5):
    for pos in range(digits - 1, -1, -1):             # last digit first
        bins = [[] for _ in range(10)]
        for code in codes:                            # stable: each bin keeps arrival order
            bins[int(code[pos])].append(code)
        codes = [c for b in bins for c in b]          # collect the bins in order
        print(f"after digit {pos + 1}:", codes)
    return codes

letters = ["78464", "10115", "78462", "80331", "78457", "10117", "50667", "78464"]
result = radix_sort(letters)
print(result == sorted(letters))

after digit 5: ['80331', '78462', '78464', '78464', '10115', '78457', '10117', '50667']
after digit 4: ['10115', '10117', '80331', '78457', '78462', '78464', '78464', '50667']
after digit 3: ['10115', '10117', '80331', '78457', '78462', '78464', '78464', '50667']
after digit 2: ['10115', '10117', '80331', '50667', '78457', '78462', '78464', '78464']
after digit 1: ['10115', '10117', '50667', '78457', '78462', '78464', '78464', '80331']
True


## Why each pass must be stable

If a pass reordered equal digits, it would destroy the work of the previous
passes. Watch what happens with an unstable pass:

In [2]:
def radix_sort_unstable(codes, digits=5):
    for pos in range(digits - 1, -1, -1):
        bins = [[] for _ in range(10)]
        for code in codes:
            bins[int(code[pos])].insert(0, code)      # prepend: reverses arrival order
        codes = [c for b in bins for c in b]
    return codes

print(radix_sort_unstable(letters) == sorted(letters))

False


## Linear time

$d$ passes over $n$ letters: $O(d \cdot n)$, with $d = 5$ fixed. Compare with
a comparison sort on 300 000 codes:

In [3]:
import random, time
rnd = random.Random(2)
big = [f"{rnd.randrange(100000):05d}" for _ in range(300_000)]

def radix_quiet(codes, digits=5):
    for pos in range(digits - 1, -1, -1):
        bins = [[] for _ in range(10)]
        for code in codes:
            bins[int(code[pos])].append(code)
        codes = [c for b in bins for c in b]
    return codes

t0 = time.perf_counter(); r = radix_quiet(big);  t_radix = time.perf_counter() - t0
t0 = time.perf_counter(); s = sorted(big);       t_sort  = time.perf_counter() - t0
print(r == s, f"radix {t_radix:.2f} s   sorted() {t_sort:.2f} s  (sorted() is C code; the point is the growth rate)")

True radix 0.15 s   sorted() 0.10 s  (sorted() is C code; the point is the growth rate)


**What we used:** stable sorting (session 7, section 3), and the comparison
lower bound that radix sort escapes by not comparing (section 9).